In [1]:
import numpy as np
import pandas as pd
import yaml, os
import time

from BuildFunctions import generate_inp_file

from BuildFunctions.BB_DT import BB_DT
from BuildFunctions.Target_Horn import Target_Horn
from BuildFunctions.Baffle import Baffle
from BuildFunctions.Collimator import Collimator
from BuildFunctions.Magnet import Magnet
from BuildFunctions.Bdump import Bdump
from BuildFunctions.t2r import T2R
from BuildFunctions.Quad import Quad


# Load yaml file

In [2]:
with open('Others_inp/ESSnuSB.yaml', "r") as f:
    config = yaml.safe_load(f)

# Create cards

In [3]:
# Dictionnary where the cards are stored before saving
Cards = {"Bodies_cards"       : f"""""",
         "Regions_cards"      : f"""""",
         "Materials_cards"    : f"""""",
         "Transforms_cards"   : f"""""", 
         "Lattices_cards"     : f""""""}



# List of Element's instance -> generalised methods
elements = [BB_DT(config), 
            Target_Horn(config), 
            Baffle(config),
            Collimator(config),  
            Magnet(config),
            Bdump(config),
            T2R(config),
            Quad(config)]

# Creates the cards in each instance and add them to the dictionnary
for elmt in elements:
    print(elmt)
    elmt.create_cards()
    Cards["Bodies_cards"] += elmt.bodies
    Cards["Regions_cards"] += elmt.regions
    if elmt.DRAW == True:
        Cards["Regions_cards"] = elmt.add_regions(Cards["Regions_cards"])
    
    Cards["Materials_cards"] += elmt.materials
    Cards["Transforms_cards"] += elmt.transforms
    Cards["Lattices_cards"] += elmt.lattices
    time.sleep(1)

## Saving cards - input files

In [4]:
# README

# For reasons that are not fully understood, when including an external .inp file
# using FLAIR, the content of the included file may be deleted.

# Solution:
# All input files should be included directly inside FLAIR using the
# #include / #endinclude mechanism.

# Please double-check that the correct file is actually included, as FLAIR may
# sometimes include a different file than the one selected.
# If this happens, repeat the operation until the correct file is included.

# Note:
# Inside the #include / #endinclude block, the content may appear empty in FLAIR
# (the file content is temporarily removed from the display).

# To restore the input files:
# - Save the FLAIR project (once the includes are set).
# - Regenerate the input files (for example using this notebook).
# - Reload the project when FLAIR asks for it.
#   If no prompt appears, reload the project manually by reopening it.

In [5]:
inp_names = ["ESSnuSBp_bodies", "ESSnuSBp_regions", 
             "ESSnuSBp_materials", "ESSnuSBp_ROTDEFI",
             "ESSnuSBp_lattices"]

for cards, inp_name in zip(Cards.values(), inp_names):
    generate_inp_file(f'{inp_name}.inp', cards)
    os.chmod(f'Others_inp/{inp_name}.inp', 0o744) 

 --- created file ESSnuSBp_bodies.inp with 130 lines
 --- created file ESSnuSBp_regions.inp with 94 lines
 --- created file ESSnuSBp_materials.inp with 59 lines
 --- created file ESSnuSBp_ROTDEFI.inp with 24 lines
 --- created file ESSnuSBp_lattices.inp with 20 lines


In [6]:
# The remaining code is not meant to be run 
# So here I break the code (in case restart and run everything)
breakline = 89jnxc

SyntaxError: invalid imaginary literal (3709055380.py, line 3)

# Others & Tests

Define ROT-DEFI
#### GEOBEGIN  
Define BODIES  
Define REGIONS  
Define LATTICE
#### GEOEND
DEFINE MAT  


#### Workflow
Fully define Model (Bodies, Regions, Materials) in parking  
Define Trasformation, if needed  
Define Body of the Volume where you want to place the replicas (applying  needed transormfation)  
Define LATTICE -> Region aimed, Model to use and transofmation needed.  

In [ ]:
x1 = 800
y1 = -250

x2 = 1028
y2 = -379

dx = x2-x1
dy = y2-y1


np.degrees(np.arctan(dy / dx))

In [ ]:
data = np.loadtxt("Bz_1540A.txt")

x_data1540 = (data[:, 0] + 4.085)
y_data1540 = data[:, 1]


data = np.loadtxt("Bz_1340A.txt")

x_data1340 = (data[:, 0] + 4.085)
y_data1340 = data[:, 1]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def gauss(x, mu, sigma, A):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2))


# Fit 1540
popt1540, pcov = curve_fit(gauss, x_data1540, y_data1540, p0=[x_data1540.mean(), 0.4, 0.4])
mu_fit1540, sigma_fit1540, A_fit1540 = popt1540
x_fit1540 = np.linspace(min(x_data1540), max(x_data1540), len(x_data1540))
y_fit1540 = gauss(x_fit1540, mu_fit1540, sigma_fit1540, A_fit1540)

# Fit 1340
popt1340, pcov = curve_fit(gauss, x_data1340, y_data1340, p0=[x_data1340.mean(), 0.4, 0.4])
mu_fit1340, sigma_fit1340, A_fit1340 = popt1340
x_fit1340 = np.linspace(min(x_data1340), max(x_data1340), len(x_data1340))
y_fit1340 = gauss(x_fit1340, mu_fit1340, sigma_fit1340, A_fit1340)


y_76T = gauss(x_fit1540, mu_fit1540, sigma_fit1540, 0.76)

plt.figure(figsize=(7,6))


plt.plot(x_data1540, y_data1540, "k+", markersize = 10, label="Tamer 1540A")
plt.plot(x_fit1540, y_fit1540, "r-")

plt.plot(x_data1340, y_data1340, "b+", markersize = 10, label="Tamer 1340A")
plt.plot(x_fit1340, y_fit1340, "r-", label="Guassian fit")

plt.plot(x_fit1540, y_76T, "y-", label="A = 0.76")


xtxt = mu_fit1540 - 2.8*np.abs(sigma_fit1540)
plt.text(xtxt + 0, 0.7, r"$\mathcal{G}$ = A exp($-\frac{(x-\mu)^2}{2\sigma^2}$)", size = 16)
plt.text(xtxt + 5.6*np.abs(sigma_fit1540), 0.67, f"Fit parameters", size = 14, horizontalalignment = "right")
plt.text(xtxt  + 4.5*np.abs(sigma_fit1540), 0.53, 
         f"1540A \n" +r"$\mu$ = " + f"{mu_fit1540:.3f}\n" + r"$\sigma$ = " + f"{sigma_fit1540:.3f}\n" + r"A =" + f"{A_fit1540:.3f}",
         horizontalalignment = "left", size = 12)

plt.text(xtxt  + 4.5*np.abs(sigma_fit1540), 0.37, 
         f"1340A \n" +r"$\mu$ = " + f"{mu_fit1340:.3f}\n" + r"$\sigma$ = " + f"{sigma_fit1340:.3f}\n" + r"A =" + f"{A_fit1340:.3f}",
         horizontalalignment = "left", size = 12)


plt.legend()
plt.xlabel("z [m]")
plt.ylabel("By [T]")

plt.show()

In [ ]:
plt.figure(figsize=(7,6))


plt.plot(x_data1540, 100 * np.abs(y_data1540 - y_fit1540)/y_data1540, "r", markersize = 10, label="1540A")
plt.plot(x_data1340, 100 * np.abs(y_data1340 - y_fit1340)/y_data1340, "b", markersize = 10, label="1340A")

plt.legend()
plt.xlabel("z [m]")
plt.ylabel(r"$\Delta$By / By [%]")

plt.show()

In [ ]:
mu = 525
sigma = 41
A = 1.1
y = gauss(x_fit1540 * 100, mu, sigma, A)

x = (x_fit1540) * 100

x *= 2.5
x -= 787.5

plt.figure()

plt.plot(x, y)

plt.plot([x.mean() - 0.75*100, x.mean() - 0.75*100], [-1, 1], 'k--')
plt.plot([x.mean() + 0.75*100, x.mean() + 0.75*100], [-1, 1], 'k--')

plt.ylim(-0.05, 1.3)

plt.show()